# Experiment — Shuffled-Pairing EOG→EEG Regression

This experiment evaluates the effect of breaking the original trial-level
correspondence between joint horizontal and vertical EOG signals and
their EEG targets.

The shuffled-pairing EOG model uses EOG signals from a different trial
of the same participant, within the same cross-validation partition,
to predict the original EEG target trial.

The experiment follows the same derangement procedure, GroupKFold
partitions, and fold-specific random-seed scheme as the shuffled-pairing
video model.

## Conditions

1. EOG model: original trial-matched hEOG/vEOG predictors.
2. Shuffled-pairing model, original-target scoring.
3. Shuffled-pairing model, permuted-target scoring.

Conditions 2 and 3 use the same fitted shuffled-pairing EOG model
predictions. They differ only in the EEG reference array used during
evaluation. Both scoring conditions are implemented in the separate
analysis notebook.

## Experimental scope

- Participants: 18 (`VALID_SUBJECTS`)
- EEG channels: Fz, Pz, Oz
- EOG predictors: hEOG and vEOG
- Trials: 40 per participant
- Trial duration: 60 seconds
- Sampling frequency: 128 Hz
- Cross-validation: 5-fold GroupKFold
- Training trials: 32 per fold
- Testing trials: 8 per fold

## Shuffled-pairing procedure

Training and testing EOG assignments are independently shuffled
within their respective CV partitions.

Every EEG target trial is assigned EOG signals from a different trial
of the same participant and the same CV partition.

No fixed points are allowed. The original EEG target trials are never
permuted during model fitting.

One deterministic shuffled pairing is generated per fold, using the
same fold-specific seed scheme as the video shuffled-pairing control.

## Preprocessing

- DEAP preprocessed Python `.dat` arrays
- hEOG and vEOG
- Fourth-order Butterworth band-pass filtering
- Frequency range: 4–45 Hz
- Forward-backward zero-phase filtering
- Fold-specific `StandardScaler`, fitted using training trials only

## Regression

- Model: scikit-learn `LinearRegression`
- Predictors: hEOG and vEOG jointly
- One model is fitted independently for each fold
- No neural network is trained

## Output

One set of held-out predictions and trial mappings is stored for each
participant and EEG channel. Both original-target and permuted-target
scoring are performed later in the analysis notebook.

Output directory: `EOG-Shuffled/Results/`

## 1. Environment Setup and Project Imports

In [ ]:
# %%
import os
import sys
import pickle
import traceback
import numpy as np

# ------------------------------------------------------------
# Environment
# ------------------------------------------------------------

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

# ------------------------------------------------------------
# Project paths
# ------------------------------------------------------------

if IN_COLAB:
    SOURCE_ROOT = '/content/drive/MyDrive/Colab Notebooks/Face2EEG/Source'
    DATA_ROOT = '/content/drive/MyDrive/Colab Notebooks/Face2EEG/Data'
else:
    SOURCE_ROOT = r'D:\Thesis\Codes\Source'
    DATA_ROOT = r'D:\Thesis\Codes\Data'

if not os.path.isdir(os.path.join(SOURCE_ROOT, 'face2eeg')):
    raise FileNotFoundError(
        f"face2eeg package not found under {SOURCE_ROOT}"
    )

if not os.path.isdir(DATA_ROOT):
    raise FileNotFoundError(f"Data directory not found: {DATA_ROOT}")

if SOURCE_ROOT not in sys.path:
    sys.path.insert(0, SOURCE_ROOT)

# ------------------------------------------------------------
# Project imports
# ------------------------------------------------------------

from face2eeg.core.experiments import run_shuffled_eog_regression
from face2eeg.core.config import SEED, FS_EEG, VALID_SUBJECTS
from face2eeg.core.data import clear_signals_cache

print("Environment:", "Google Colab" if IN_COLAB else "Local")
print("Source root:", SOURCE_ROOT)
print("Data root:", DATA_ROOT)
print("EEG sampling rate:", FS_EEG)
print("Base seed:", SEED)

## 2. Experiment Configuration

In [ ]:
# %%
# ------------------------------------------------------------
# Output directories
# ------------------------------------------------------------

BASE_DIR = os.path.join(
    DATA_ROOT, 'Experiments', 'Cross-Validation'
)
EXP_DIR = os.path.join(BASE_DIR, 'EOG-Shuffled')

# ------------------------------------------------------------
# Execution mode
# ------------------------------------------------------------

SMOKE_TEST = False

if SMOKE_TEST:
    SUBJECTS = [2]
    CHANNELS = ['Oz']
    RESULTS_DIR = os.path.join(EXP_DIR, 'Smoke-Test', 'Results')
else:
    SUBJECTS = list(VALID_SUBJECTS)
    CHANNELS = ['Fz', 'Pz', 'Oz']
    RESULTS_DIR = os.path.join(EXP_DIR, 'Results')

os.makedirs(RESULTS_DIR, exist_ok=True)

# ------------------------------------------------------------
# Cross-validation
# ------------------------------------------------------------

N_POINTS = 40
N_SPLITS = 5

# ------------------------------------------------------------
# EOG predictors and preprocessing
# ------------------------------------------------------------

REFERENCES = ('heog', 'veog')
FILTER_BAND = (4.0, 45.0)

# ------------------------------------------------------------
# Randomization
# ------------------------------------------------------------

SHUFFLE_SEED = int(SEED)

# ------------------------------------------------------------
# Configuration checks
# ------------------------------------------------------------

if not set(SUBJECTS).issubset(set(VALID_SUBJECTS)):
    raise ValueError("SUBJECTS contains invalid participant IDs.")

if not set(CHANNELS).issubset({'Fz', 'Pz', 'Oz'}):
    raise ValueError("Invalid EEG channel selection.")

if N_POINTS != 40 or N_SPLITS != 5:
    raise ValueError("This experiment is configured for 40 trials and 5 folds.")

if FS_EEG != 128:
    raise ValueError(f"Expected FS_EEG=128 Hz; received {FS_EEG}.")

print("Experiment directory:", EXP_DIR)
print("Results directory:", RESULTS_DIR)
print("Smoke test:", SMOKE_TEST)
print("Subjects:", SUBJECTS)
print("Channels:", CHANNELS)
print("Trials:", N_POINTS)
print("CV folds:", N_SPLITS)
print("Training trials per fold:", N_POINTS - N_POINTS // N_SPLITS)
print("Testing trials per fold:", N_POINTS // N_SPLITS)
print("EOG predictors:", REFERENCES)
print("EOG filter band:", FILTER_BAND, "Hz")
print("Sampling frequency:", FS_EEG, "Hz")
print("Shuffle seed:", SHUFFLE_SEED)

## 3. Existing-Result Validation

Before running an experiment, the notebook checks whether a result
file already exists and whether it matches the current experiment
configuration.

Results from the previous repeated-randomization implementation
are not treated as compatible with this single-derangement experiment.

In [ ]:
# %%
def validate_existing_result(path, subject_id, channel_name):
    """Check whether an existing result matches the current experiment."""
    try:
        with open(path, 'rb') as f:
            existing = pickle.load(f)
    except Exception as exc:
        return False, f"Could not read result file: {exc}"

    if existing.get('subject') != subject_id:
        return False, "Participant ID mismatch."

    if existing.get('channel') != channel_name:
        return False, "Channel mismatch."

    if existing.get('time_window') != 60:
        return False, "Trial duration mismatch."

    if existing.get('n_splits') != N_SPLITS:
        return False, "Number of CV folds mismatch."

    config = existing.get('config', {})

    expected_config = {
        'method': 'LinearRegression',
        'control': 'shuffled_pairing',
        'shuffle_pairing': True,
        'shuffle_seed': SHUFFLE_SEED,
        'n_points': N_POINTS,
        'n_splits': N_SPLITS,
        'references': list(REFERENCES),
        'eog_filter_band_hz': list(FILTER_BAND),
    }
    
    for key, expected_value in expected_config.items():
        if config.get(key) != expected_value:
            return False, f"Configuration mismatch: {key}."

    folds = existing.get('folds')

    if not isinstance(folds, dict) or len(folds) != N_SPLITS:
        return False, "Invalid or incomplete fold structure."

    expected_samples = 60 * FS_EEG

    for fold_idx in range(N_SPLITS):
        if fold_idx not in folds:
            return False, f"Missing fold {fold_idx}."

        fold = folds[fold_idx]

        required_fields = [
            'y_test', 'y_pred', 'trials',
            'train_trials', 'test_trials',
            'train_eog_source_trials', 'test_eog_source_trials',
            'train_permutation', 'test_permutation',
        ]

        if any(key not in fold for key in required_fields):
            return False, f"Incomplete results in fold {fold_idx}."

        train = np.asarray(fold['train_trials'])
        test = np.asarray(fold['test_trials'])
        train_source = np.asarray(fold['train_eog_source_trials'])
        test_source = np.asarray(fold['test_eog_source_trials'])

        if len(train) != 32 or len(test) != 8:
            return False, f"Unexpected partition sizes in fold {fold_idx}."

        if not np.array_equal(fold['trials'], test):
            return False, f"Target trial IDs mismatch in fold {fold_idx}."

        if not np.array_equal(np.sort(train), np.sort(train_source)):
            return False, f"Invalid training EOG mapping in fold {fold_idx}."

        if not np.array_equal(np.sort(test), np.sort(test_source)):
            return False, f"Invalid testing EOG mapping in fold {fold_idx}."

        if np.any(train == train_source) or np.any(test == test_source):
            return False, f"Fixed point detected in fold {fold_idx}."

        if fold.get('train_seed') != SHUFFLE_SEED + fold_idx:
            return False, f"Training seed mismatch in fold {fold_idx}."

        if fold.get('test_seed') != SHUFFLE_SEED + 1000 + fold_idx:
            return False, f"Testing seed mismatch in fold {fold_idx}."

        y_test = np.asarray(fold['y_test'])
        y_pred = np.asarray(fold['y_pred'])

        expected_shape = (len(test), expected_samples)

        if y_test.shape != expected_shape or y_pred.shape != expected_shape:
            return False, f"Prediction shape mismatch in fold {fold_idx}."

    return True, "Compatible result."

## 4. Run Shuffled-Pairing-Trial EOG Regression

The experiment is executed once for each participant and EEG channel.
Each fold uses independent training and testing derangements, with
the same seed scheme as the shuffled-pairing video control.

In [ ]:
# %%
for subject_idx, subject_id in enumerate(SUBJECTS, start=1):
    print("\n" + "#" * 65)
    print(f"SUBJECT {subject_id:02d} ({subject_idx}/{len(SUBJECTS)})")
    print("#" * 65)

    try:
        for channel in CHANNELS:
            print(f"\nSubject {subject_id} | Channel {channel}")

            result_path = os.path.join(
                RESULTS_DIR,
                f'sub{subject_id:02d}_ch{channel}.pkl'
            )

            # Skip only results that match the current configuration.
            if os.path.isfile(result_path):
                compatible, message = validate_existing_result(
                    result_path, subject_id, channel
                )

                if compatible:
                    print(f"[SKIP] {message} Existing result: {result_path}")
                    continue

                raise RuntimeError(
                    f"An incompatible result already exists: {result_path}\n"
                    f"Reason: {message}\n"
                    "Use a new output directory or remove the old file "
                    "deliberately before rerunning."
                )

            # Execute one shuffled-pairing experiment.
            try:
                run_shuffled_eog_regression(
                    subject_id=subject_id,
                    channel_name=channel,
                    n_points=N_POINTS,
                    n_splits=N_SPLITS,
                    references=REFERENCES,
                    fs=FS_EEG,
                    shuffle_seed=SHUFFLE_SEED,
                    filter_band=FILTER_BAND,
                    save_dir=RESULTS_DIR,
                    json_filename='EOG_Shuffled_summary.json',
                    root_dir=DATA_ROOT,
                    verbose=True,
                )

                print(f"[OK] Subject {subject_id} | {channel}")

            except Exception as exc:
                print(f"[ERROR] Subject {subject_id} | {channel}: {exc}")
                traceback.print_exc()

    finally:
        clear_signals_cache()

## 5. Verify Experiment Outputs

Check that all requested participant/channel combinations have
complete result files with the expected fold structure and pairing
mappings.

In [ ]:
# %%
print("\n" + "=" * 65)
print("Shuffled-pairing EOG REGRESSION — OUTPUT VERIFICATION")
print("=" * 65)

complete = []
incomplete = []

for subject_id in SUBJECTS:
    for channel in CHANNELS:
        path = os.path.join(
            RESULTS_DIR,
            f'sub{subject_id:02d}_ch{channel}.pkl'
        )

        if not os.path.isfile(path):
            incomplete.append((subject_id, channel, "File not found"))
            print(f"[MISSING] S{subject_id:02d} | {channel}")
            continue

        valid, message = validate_existing_result(
            path, subject_id, channel
        )

        if valid:
            complete.append((subject_id, channel))
            print(f"[OK]      S{subject_id:02d} | {channel}")
        else:
            incomplete.append((subject_id, channel, message))
            print(f"[INVALID] S{subject_id:02d} | {channel}: {message}")

print("\nCompleted:", len(complete))
print("Incomplete:", len(incomplete))
print("Results directory:", RESULTS_DIR)

if incomplete:
    print("\nSome experiments are incomplete or invalid.")
else:
    print("\nAll requested experiments have valid result files.")